## Set up basico 

In [1]:
# import dspy

# lm = dspy.LM("modelo_que_queremos_usar", api_key="YOUR_OPENAI_API_KEY")

# dspy.configure(lm=lm)

In [3]:
import os
import importlib
import lm_config as dspy_lm_config

from lm_config import configure_lm
import dspy


os.environ["DSPY_LM_BACKEND"] = "ollama"
os.environ["OLLAMA_BASE_URL"] = "http://ollama:11434"
# os.environ["OLLAMA_MODEL"] = "llama3.2:latest"
os.environ["OLLAMA_MODEL"] = "phi3:latest"

importlib.reload(dspy_lm_config)


lm = configure_lm(max_tokens=1024)
print(lm.model)

print(dspy.Predict("tema: str -> idea: str")(tema="RAG con Ollama").idea)

ollama_chat/phi3:latest
"Organiza una sesión de RAG en la que se utilice Ollama para generar música y ritmo impulsados por inteligencia artificial. La sesión puede incluir un elemento educativo, como explicar los fundamentos del diseño musical o la tecnología detrás de Ollama."


## Hola mundo - Predicciones simples

Vamos a comenzar con el ejemplo más simple posible utilizando `dspy.Predict`.

La idea es definir una tarea simple:
- input → pregunta
- output → respuesta

DSPy se encargará de construir internamente el prompting necesario.

In [ ]:
qa = dspy.Predict("pregunta: str -> respuesta: str")
r = qa(pregunta="En una frase, Que es DSPy?")
print(r.respuesta)


DSPy es un módulo de señal digital diseñado para la simulación y el análisis.


In [8]:
texto = "ingles -> espanol"
traducir = dspy.Predict(texto)
resultado = traducir(ingles="The security deposit equals two months of rent.")
print(resultado.espanol)

El depósito de seguridad equivale a dos meses de alquiler. (The security deposit equals two months of rent.)
```


In [9]:
reescribir = dspy.Predict("texto, tono -> reescritura")
r = reescribir(
    texto="Si no pagás a tiempo, hay recargo.",
    tono="formal y educado",
)
print(r.reescritura)

Se le aplicará un cargo adicional si la factura no se paga a tiempo.


- <code style="background:#fef08a; color:#1a1a1a">ingles -> espanol</code>
- <code style="background:#fdba74; color:#1a1a1a">texto, tono -> reescritura</code>
- <code style="background:#a5f3fc; color:#1a1a1a">pregunta: str -> respuesta: str</code>

Estas líneas definen una <strong style="color:#c2410c">Signature</strong>,
uno de los conceptos fundamentales de DSPy:
describen una tarea por sus entradas y salidas,
como la firma de una función tradicional.

En DSPy los nombres de los campos importan: además de definir la interfaz, le dan al modelo pistas sobre la intención. Si la salida se llama `clausula_resumida` en lugar de `respuesta`, el modelo tenderá a resumir una cláusula — y el valor quedará en `result.clausula_resumida`.


Para convertir nuestra Signature en una función ejecutable utilizamos <strong style="color:#c2410c">dspy.Predict</strong>.

Predict es uno de los Modules que ofrece DSPy. Mientras que las Signatures describen <code style="background:#a5f3fc; color:#1a1a1a">qué queremos obtener</code>, los Modules definen <code style="background:#fdba74; color:#1a1a1a">cómo vamos a conseguirlo</code>. Son los encargados de implementar la estrategia de ejecución, controlar el flujo del programa, utilizar herramientas adicionales y gestionar otros aspectos de la interacción con el modelo.

_____________________


Cuando hacemos:

```python
qa = dspy.Predict("pregunta: str -> respuesta: str")
```

ocurren el siguiente flujo


## Flujo Signature → Predict

```mermaid
flowchart TD

A["pregunta: str -> respuesta: str"]
    --> B["DSPy parsea la Signature"]

B --> C["Signature<br/>Input: pregunta : str<br/>Output: respuesta : str"]

C --> D["Genera las instrucciones"]

D --> E["Dado el campo 'pregunta'<br/>produce el campo 'respuesta'."]

E --> F["Crea el módulo Predict"]

F --> G["qa"]
```


1. Convierte la cadena `"pregunta: str -> respuesta: str"` en una clase `Signature`, donde tanto `pregunta` como `respuesta` tienen como tipo predeterminado `str`.
2. Genera una instrucción para el modelo de lenguaje. En este caso sería:

```text
Given the field `pregunta`, produce the field `respuesta`.
```

3. Crea una instancia del módulo `Predict` utilizando esa `Signature`.

A partir de ese momento, `qa` puede utilizarse como cualquier otra función de Python.

___________

Cuando ejecutamos:

```python
result = haiku_generator(subject="computer science")
```

DSPy realiza internamente el siguiente proceso:

## Flujo de ejecución de `Predict`

```mermaid
flowchart LR

subgraph USER["Tu código"]
    A["qa(pregunta='¿Qué es DSPy?')"]
    H["r.respuesta"]
end

subgraph DSPY["DSPy"]
    B["Verificar LM"]
    C["ChatAdapter"]
    D["Construir Prompt"]
    E["Parsear respuesta"]
    F["Prediction"]
end

subgraph MODEL["LLM"]
    G["Ollama"]
end

A --> B
B --> C
C --> D
D --> G
G --> E
E --> F
F --> H
```

1. Verifica que exista un modelo de lenguaje (LM) configurado.
2. Utiliza un **Adapter** para transformar la `Signature` y sus entradas en un formato que el modelo pueda comprender.
3. El `ChatAdapter` (el adaptador utilizado por defecto) construye automáticamente el prompt incluyendo:
   - Las instrucciones generadas a partir de la `Signature`.
   - El esquema de los campos de entrada y salida.
   - Las instrucciones de formato.
   - Los valores proporcionados por el usuario.
4. Envía los mensajes al modelo de lenguaje.
5. Recibe la respuesta del modelo.
6. Analiza la respuesta y extrae los campos definidos en la `Signature`.
7. Devuelve un objeto `Prediction`, cuyos valores pueden accederse directamente desde Python.